Multi-fidelity Modeling and Experimental Design (Active Learning)

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None)
from numpy.random import default_rng
import os
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
resum_path = os.getenv("RESUM_PATH")

In [124]:
bounds = {"radius": [90, 265], "thickness": [1,5],"npanels": [4,360], "theta": [0,np.pi/2], "length": [1,150] }
num_samples = len(bounds)*1
num_samples_1 = len(bounds)*1
num_samples_2 = len(bounds)*2
num_digits = 2

In [24]:
from scipy.optimize import minimize


def get_outer_radius(coeff):
    theta=coeff[3] * np.pi/180.
    l = coeff[4]/2

    Ax = coeff[0] + (l*np.cos(np.pi/2 - theta)) + np.cos(theta*np.pi/180.)* coeff[1]/2.
    Ay = l*np.sin(np.pi/2 - theta)          + np.sin(theta*np.pi/180.)* coeff[1]/2.
    Bx = coeff[0] - (l*np.cos(np.pi/2 - theta)) + np.cos(theta*np.pi/180.)* coeff[1]/2.
    By = -l*np.sin(np.pi/2 - theta)         + np.sin(theta*np.pi/180.)* coeff[1]/2.

    def outer_radius(x, Ax, Bx, Ay, By):
        y1=(x-Ax)*(By-Ay)/(Bx-Ax)+Ay
        return -1.*np.sqrt(x**2+y1**2)

    x0 = Ax
    bounds=[[np.min([Ax,Bx]),np.max([Ax,Bx])]]
    res = minimize(outer_radius, x0, args=(Ax,Bx,Ay,By), bounds=bounds)
    return np.abs(res.fun)

def get_inner_radius(coeff):
    theta=coeff[3] * np.pi/180.
    l = coeff[4]/2
        
    Bx = coeff[0] + (l*np.cos(np.pi/2 - theta)) - np.cos(theta*np.pi/180.)* coeff[1]/2.
    By = l*np.sin(np.pi/2 - theta)          - np.sin(theta*np.pi/180.)* coeff[1]/2.
    Ax = coeff[0] - (l * np.cos(np.pi/2 - theta)) - np.cos(theta*np.pi/180.)* coeff[1]/2.
    Ay = -l*np.sin(np.pi/2 - theta)         - np.sin(theta*np.pi/180.)* coeff[1]/2.

    def inner_radius(x, Ax, Bx, Ay, By):
        y1=(x-Ax)*(By-Ay)/(Bx-Ax)+Ay
        return np.sqrt(x**2+y1**2)

    x0 = Ax
    bounds=[[np.min([Ax,Bx]),np.max([Ax,Bx])]]
    res = minimize(inner_radius, x0, args=(Ax,Bx,Ay,By), bounds=bounds)
    return res.fun

In [125]:


def get_random_samples(bounds, num_samples=9, seed=None):
    rng = default_rng(seed)
    samples = {}
    for parameter, bound in bounds.items():
        samples[parameter] = rng.uniform(bound[0], bound[1], num_samples)
        samples[parameter] = np.round(samples[parameter],num_digits)
    return pd.DataFrame(samples)

random_samples = get_random_samples(bounds, 5,seed=None)
random_samples

,radius,thickness,npanels,theta,length
0,234.38,3.55,283.67,0.99,68.31
1,185.86,3.28,172.25,1.47,143.21
2,259.57,2.29,312.45,1.05,77.26
3,115.65,1.38,262.30,0.08,101.66
4,177.61,2.11,327.23,0.46,119.71


ESS = $(\sum_{n=1}^Nw^t_n(\tau_t))^2/(\sum_{n=1}^N(w^t_n(\tau_t))^2)$(8)<br>
<br>
<br>
Another key step of the SCMC algorithm is the sampling step (step 4(g)) that prevents particle degeneracy. If the sampling step is skipped or is not done effectively a few probable points are repeatedly copied in the resampling step and in the end one might be left with a small proportion of distinct points in the input space. The sampling step comprises one (or a few) MCMC transition step(s) that move the samples slightly under the current posterior at time t, i.e., a proposal followed by a Metropolis-Hastings accept/reject step for each sample point. The efficiency of the algorithm can be improved by using the covariance structure of the current density in the proposal distribution if such information is available. However, as a general guideline we suggest Gibbs type transitions (i.e., one dimension at a time) with normal proposal distributions whose variances are chosen adaptively by monitoring the acceptance rate from the previous time step. The notation $x^t_n ∼ K^t$ is used to show the Gibbs/Metropolis-Hastings step for a sample point $x_n$ where $K^t$ is a transition kernel for $π_t$.
<br>
<br>
https://blog.djnavarro.net/posts/2023-04-12_metropolis-hastings/
<br>
The algorithm proceeds as follows: Given an initial value x(0), for each j=1,2,…,N<br>
Draw y from q(x(j),⋅) and u from U(0,1)<br>
If u≤α(x(j),y), set x(j+1)=y.<br>
Otherwise, set x(j+1)=x(j)<br>
<br>
α(x,y)=π(y)q(y,x)/π(x)q(x,y).
<br>
<br>
SCMC sampling from space X <br>
Inputs: Hypercube $Q_d$ ⊃ X, constraint $C_X$, sample size N, Target constraint parameter value $τ_T$. <br>
1: t ← 0 <br>
2: Generate a uniform sample S of size N on $Q_d$; <br>
3: Initiate the weights $W^0_n$ ← 1/N for $n = 1,...,N$; <br>
4: while $τ_t ≤ τ_T$ do <br>
(a) t ← t+1 <br>
(b) Numerically solve (8) to obtain τt; <br>
(c) $W^t_n$ ← $W^{t−1}_n w^t_n$ where $w^t_n = Φ(−τ_tC_X(x_{n}^{t−1} ))/Φ(−τ_{t−1}C_X(x_{n}^{t−1})) , n = 1,...,N$; <br>
(d) Normalize $W^t_{1:N}$, i.e., $W^t_n ← W^t_n/(\sum_{n=1}^N W^t _n)$ ; <br>
(e) Resample $x^{t−1}_{1:N}$ with weights $W^t_{1:N}$; <br>
(f) $W^t_{1:N}$ ← 1/N ; <br>
(g) Sample $x^t_{1:N} ∼ K^t$ (refer to the text for $K^t$); <br>
5: end while<br>
Return: Uniform sample $x^T_{1:N}$ from X.

In [70]:
from statsmodels.tsa.stattools import acf as autocorr
def neff(arr):
    n = len(arr)
    acf = autocorr(arr, nlags=n, fft=True)
    sums = 0
    for k in range(1, len(acf)):
        sums = sums + (n-k)*acf[k]/n

    return n/(1+2*sums)

In [105]:
from scipy.stats import norm

def Condition_x(x):
    Cx=[]
    for i in x:
        Cx.append([get_inner_radius(i)-90.])
        #Cx.append([get_inner_radius(i)-90.,get_outer_radius(i)-265.,get_outer_radius(i)-get_inner_radius(i)-20.,i[2]*i[1]*i[4]-np.pi*(get_outer_radius(i)**2-get_inner_radius(i)**2)])
    
    return np.array(Cx)
    #return 
def get_weights_test(tau_1,tau_2,Cx):
    arg1=np.ones(len(Cx))
    arg2=np.ones(len(Cx))

    for i in range(len(Cx)):
        for j in range(len(Cx[i])):
            print(arg1, Cx[i][j], norm.cdf(Cx[i][j]),  norm.cdf(-tau_1*Cx[i][j]))
            arg1 *= norm.cdf(-tau_1*Cx[i][j])
            #print(arg1)
            arg2 *= norm.cdf(-tau_2*Cx[i][j])
        
    return arg2/arg1


In [126]:
samples = []
N=100
n=0
while n < N:
    s = get_random_samples(bounds, 1,seed=None).to_numpy()[0]
    #print(s,get_inner_radius(s),get_outer_radius(s), get_outer_radius(s)-get_inner_radius(s),s[2]*s[1]*s[4])
    if get_inner_radius(s) < 90.:
        continue
    elif get_outer_radius(s) > 265.:
        continue
    elif get_outer_radius(s)-get_inner_radius(s)  > 20.:
        continue
    elif s[2]*s[1]*s[4] > np.pi*(get_outer_radius(s)**2-get_inner_radius(s)**2):
            continue
    else:
        samples.append(s)
        n = len(samples)
    
df=pd.DataFrame(samples,columns=bounds.keys())
print(df)



/var/folders/99/0svbmlns6xs9l9p55lcr912r0000gn/T/ipykernel_81012/2571105852.py:32: RuntimeWarning: invalid value encountered in divide
  y1=(x-Ax)*(By-Ay)/(Bx-Ax)+Ay
/var/folders/99/0svbmlns6xs9l9p55lcr912r0000gn/T/ipykernel_81012/2571105852.py:14: RuntimeWarning: invalid value encountered in divide
  y1=(x-Ax)*(By-Ay)/(Bx-Ax)+Ay


    radius  thickness  npanels  theta  length
0   222.89       1.61    35.81   0.95   49.07
1   152.02       2.03    55.01   0.32   20.18
2   166.25       3.00    32.09   0.50   18.67
3   233.10       1.80   155.73   0.63    9.82
4   181.96       4.89    11.32   0.99    4.42
5   256.09       2.53    45.07   1.20   41.60
6    92.04       2.09    27.90   0.48   97.82
7   190.77       1.69    63.01   0.62   14.86
8   137.23       1.36    22.37   0.82  137.47
9   193.27       3.46    13.55   1.51   54.67
10  184.25       4.16   124.61   0.11    4.49
11  257.99       3.10   166.13   0.35    9.54
12  222.35       1.11    72.58   0.53   21.04
13  195.14       3.23     6.72   0.74   41.21
14  227.60       4.73    10.71   0.04  138.78
15  199.05       3.60    44.91   1.16   17.42
16  237.41       3.06    23.76   0.68    6.96
17  199.75       1.07    98.17   1.36  105.68
18  204.59       2.52   161.18   0.26    4.95
19  157.32       1.84    77.03   1.28    7.82
20  219.72       4.11    28.28   1

In [106]:
N=5
tau_t1= tau_t2 = 0.
tau_T = 1.e6
t=0
random_samples = get_random_samples(bounds, N,seed=None)
x_t1=random_samples.to_numpy()

W_t1=np.array([1/N for i in range(N)])

print(get_weights_test(1.,10.,Condition_x(x_t1)))

[1. 1. 1. 1. 1.] 18.755627244060406 1.0 8.707550427388503e-79
[8.70755043e-79 8.70755043e-79 8.70755043e-79 8.70755043e-79
 8.70755043e-79] 106.68299679124473 1.0 0.0
[0. 0. 0. 0. 0.] -0.283290602885657 0.3884770400902572 0.6115229599097428
[0. 0. 0. 0. 0.] 14.068131060725435 1.0 2.981219355788452e-45
[0. 0. 0. 0. 0.] 55.04764467724388 1.0 0.0
[nan nan nan nan nan]


/var/folders/99/0svbmlns6xs9l9p55lcr912r0000gn/T/ipykernel_81012/1595203336.py:22: RuntimeWarning: invalid value encountered in divide
  return arg2/arg1


-14208267858.779495

In [41]:
from scipy.optimize import fsolve

N=5
tau_t1= tau_t2 = 0.
tau_T = 1.e6
t=0
random_samples = get_random_samples(bounds, N,seed=None)
x_t1=random_samples.iloc[0].to_numpy()
W_t1=np.array([1/N for i in range(N)])
print(Condition_x(x_t1))
while tau_t2 <= tau_T:
    t = t+1
    ESS = neff(x_t1)

    func = lambda tau : get_weights(tau,tau_t1,Condition_x(x_t1)-ESS)
    tau_solution = fsolve(func, 0.1)
    #w_t = get_weights(tau_t2,tau_t1,Condition_x(x_t1))
    #for i in range(N):
    #    ESS = np.sum(w_t)^2/np.sum(w_t**2)
    
   # W_t2 = W_t1 * w_t
    #normalize
    #W_t2 = [W_t2[i]/np.sum(W_t2) for i in range(N)]
    #W_t1 = W_t2
    # Metropolis Hasting Step



8.933316595376363


KeyboardInterrupt: 

In [ ]:
### Metropolis-Hastings MCMC algorithm

# 'target' is a probability distribution, that is, a function that sums to 1,
# here implemented as a Python function.
# 

size = 10000 # length of MCMC random sampling

walk = np.zeros(size) # array of samples 

all_proposed = np.zeros(size)
all_current = np.zeros(size)

walk[0] = 3 #initialize first step with dummy value to get MCMC walk started

# the random walk
for i in range(1,size):
    current = walk[i-1]
    all_current[i] = current
    proposed = current + pm.rnormal(0,1/ 1**2)
    all_proposed[i] = proposed
    
    A = target(proposed) / target(current) # accept ?
    
    # ratio above expresses ratio of probabilities for proposed outcome vs current outcome, accoriding to distribution
    # if ratio > 1 : accept always. if ratio < 1, accept if ratio > random number 0..1
    
    # For proposals outside domain of independent variable, i.e the outcome, for instance, if current is 5, 
    # that is, domain max of independent variable , and proposed is 6,
    # then, since all values outside of domain for the independent have p==0, that proposal is not accepted. 
    # This can be seen in the trace plot, where proposals occur outside of domain [0..5], but are never accepted. 
    
    if pm.runiform(0,1) < A : 
        walk[i] = proposed # accept
    else:
        walk[i] = current

# condition 1:
$r_{inner}$ > 90. [cm]<br>
$r_{outer}$ < 265. [cm]<br>
$r_{outer}$-$r_{inner}$ < 20. [cm] <br>
V <= $\pi\cdot(r_{outer}^2-r_{inner}^2)$<br>




In [ ]:
version='v1.4'
if not os.path.exists(f'{resum_path}/out/{version}'):
   os.makedirs(f'{resum_path}/out/{version}')
if not os.path.exists(f'{resum_path}/simulation/{version}/macros'):
   os.makedirs(f'{resum_path}/out/{version}/macros')
height=300
zpos=42
design=4
labels= ["radius", "npanels", "theta", "thickness", "length"]
bounds = {"radius": [95, 255], "npanels": [4,360], "theta": [0,45], "thickness": [1,15], "length": [1,150] }
num_samples = len(bounds)*60
num_samples_to_add = num_samples*2